# Day 9 | Lab 9.3: 3-Layer Agent Evaluation Hierarchy + Skills & Prompt Optimization

**Duration.** ~75 minutes  ·  **Difficulty.** Dense (Interview Prep included)

## Scenario
ShopSmart, a multi-channel retailer, has shipped a LangGraph customer-support agent (built in Day 5/8 labs). Quality complaints are starting to appear — sometimes the agent picks the wrong category, sometimes the response is fine but the tone is off, sometimes it gives a confident answer with the wrong order ID. Eyeballing 5 examples is no longer enough. We need a **systematic evaluation suite** that scores every release on the same yardstick.

Then, once we have the eval harness, we improve the agent itself: TechNova (a parallel ShopSmart line-of-business) wants to consolidate three specialist prompts into a **reusable skills library** so that the same base agent can serve claims, customer support, and compliance — selecting the right skill on the fly. We A/B-test the skills approach against a generic prompt, find the weakest skill, and run a **prompt-optimization loop** to fix it.

## Learning Objectives
1. Build the **3-layer agent evaluation hierarchy**: routing accuracy → response quality (LLM-as-Judge) → trajectory / tool correctness.
2. Compute **pass@k** for stochastic agents and explain when to use it.
3. Compose a labelled test dataset with `must_contain` keyword anchors as a cheap proxy for tool correctness.
4. Implement the **reusable skills as system prompts** pattern (vs tool-loaded skills).
5. Run an **A/B test** between a generic agent and a skill-routed agent, scored by LLM-as-Judge.
6. Drive a **prompt-optimization loop**: identify the weakest skill, analyse failures, patch the rules, re-run the eval.


## 1. Install Dependencies


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# pip install "langchain>=1.0" "langchain-core>=1.0" "langgraph>=1.0" \
#     "langchain-openai>=1.0" "langchain-anthropic>=0.3" \
#     pydantic matplotlib pandas tiktoken


## 2. API Key Configuration

Load `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` from `.env` (preferred) or your shell environment. The lab uses Anthropic's Claude as the **agent** model and OpenAI as the **judge** to keep judge / actor on different model families (a common debiasing tactic — see Q&A).


In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


## 3. Imports & Client Init


In [ ]:
from typing import TypedDict, Literal
from collections import Counter
import copy
import statistics
import time

import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END

# Agent model — Claude for the system under test
AGENT_MODEL = 'claude-sonnet-4-5-20250929'
JUDGE_MODEL = 'gpt-5-mini'  # different family than the agent — reduces self-preference bias

agent_llm = ChatAnthropic(model=AGENT_MODEL, temperature=0)
judge_llm = ChatOpenAI(model=JUDGE_MODEL, temperature=0)
print(f'Agent: {AGENT_MODEL}\nJudge: {JUDGE_MODEL}')


## 4. The 3-Layer Eval Hierarchy

| Layer | What it evaluates | Metric type | What it catches |
|-------|------------------|-------------|-----------------|
| **L1 — Routing** | Did the agent pick the right tool / category? | Deterministic (exact match) | Wrong handler, wrong tool, wrong agent in a multi-agent setup |
| **L2 — Response quality** | Is the answer accurate, complete, helpful, well-toned? | LLM-as-Judge with rubric | Hallucinations, missing detail, robotic tone |
| **L3 — Trajectory / tool correctness** | Did the agent take the *right steps* and use the *right inputs*? | Contains-check / span match | Looked up wrong order, called wrong API, skipped a required step |

Why three layers? A response can score well on quality (fluent, confident) while routing is wrong (sent to billing instead of order-status) — L1 catches it. A response can route correctly *and* read well while citing the wrong order ID — L3 catches it. Each layer has a job no other layer can do.


## 5. The System Under Test — ShopSmart Support Agent

We rebuild the agent from Day 5/8 in compact form. It routes a customer query into one of four categories (`order_status`, `returns`, `billing`, `product_info`) and produces a response using a small in-memory database.


In [ ]:
# --- ShopSmart Spine A Data ---
ORDERS_DB = {
    'ORD-1001': {'customer': 'Priya Sharma', 'item': 'Samsung Galaxy S24', 'amount': '₹74,999',
                 'status': 'In Transit', 'city': 'Mumbai', 'eta_days': 2,
                 'tracking': 'BLUEDART-7891234'},
    'ORD-1002': {'customer': 'Arjun Patel', 'item': 'Sony WH-1000XM5', 'amount': '₹26,990',
                 'status': 'Delivered', 'city': 'Bangalore', 'eta_days': 0,
                 'delivered_on': '2026-04-28'},
    'ORD-1003': {'customer': 'Anika Nair', 'item': 'Apple Watch Series 9', 'amount': '₹46,900',
                 'status': 'Returned', 'city': 'Pune', 'refund_status': 'Processed', 'refund_days': 5},
    'ORD-1004': {'customer': 'Vikram Singh', 'item': 'iPad Air', 'amount': '₹54,900',
                 'status': 'Pending Payment', 'city': 'Delhi', 'payment_method': 'UPI'},
}

PRODUCT_INFO = {
    'samsung galaxy s24': 'Samsung Galaxy S24 — 8GB RAM, 256GB, ₹74,999. In stock. Ships next-day.',
    'sony wh-1000xm5':    'Sony WH-1000XM5 — Industry-leading noise cancelling, ₹26,990. 30-day return.',
    'apple watch series 9':'Apple Watch Series 9 GPS — 45mm, ₹46,900. AppleCare optional.',
}


In [ ]:
# --- Compact LangGraph agent (compiled once, reused for eval) ---
class AgentState(TypedDict):
    customer_query: str
    category: str
    response: str

class RouteDecision(BaseModel):
    category: Literal['order_status', 'returns', 'billing', 'product_info']

router_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You route ShopSmart customer queries into one of: order_status, returns, billing, product_info. Reply with the category only.'),
    ('user', '{query}'),
])
router = router_prompt | agent_llm.with_structured_output(RouteDecision)

def route_node(state: AgentState) -> AgentState:
    decision = router.invoke({'query': state['customer_query']})
    return {'category': decision.category}

def order_status_node(state: AgentState) -> AgentState:
    q = state['customer_query']
    for oid, o in ORDERS_DB.items():
        if oid in q:
            r = (f"Your order {oid} for the {o['item']} is currently {o['status']}."
                 f" Expected delivery in {o.get('eta_days','—')} days to {o['city']}.")
            if 'tracking' in o: r += f" Tracking ID: {o['tracking']}."
            return {'response': r}
    return {'response': 'Please share your order ID (format ORD-1xxx) so I can check status.'}

def returns_node(state: AgentState) -> AgentState:
    q = state['customer_query']
    for oid, o in ORDERS_DB.items():
        if oid in q:
            if o['status'] == 'Returned':
                return {'response': (f"Order {oid} ({o['item']}) was returned. Refund status: {o.get('refund_status','—')}."
                                     f" Refunds typically credit in {o.get('refund_days', 5)} business days.")}
            return {'response': (f"To return order {oid} ({o['item']}, {o['amount']}): pickup is free within the 7-day window."
                                 f" Refund processes in 5 business days after pickup.")}
    return {'response': 'For returns, share the order ID. The 7-day return window starts from delivery.'}

def billing_node(state: AgentState) -> AgentState:
    q = state['customer_query']
    for oid, o in ORDERS_DB.items():
        if oid in q:
            if o['status'] == 'Pending Payment':
                return {'response': (f"Order {oid} for {o['item']} is awaiting payment ({o['amount']}, {o.get('payment_method')}). "
                                     f"Please complete the UPI mandate within 24 hours.")}
            return {'response': f"Order {oid} ({o['item']}) — invoice amount {o['amount']}. Payment status: completed."}
    return {'response': 'For billing queries please share the order ID and the issue (failed payment, double-charge, refund delay).'}

def product_info_node(state: AgentState) -> AgentState:
    q = state['customer_query'].lower()
    for name, info in PRODUCT_INFO.items():
        if name in q:
            return {'response': info}
    return {'response': 'I can share specs, price, and stock for any product in our catalogue — please name it.'}

def select_handler(state: AgentState) -> str:
    return state['category']

graph = StateGraph(AgentState)
graph.add_node('router', route_node)
for n, f in [('order_status', order_status_node), ('returns', returns_node),
             ('billing', billing_node), ('product_info', product_info_node)]:
    graph.add_node(n, f)
    graph.add_edge(n, END)
graph.set_entry_point('router')
graph.add_conditional_edges('router', select_handler,
    {'order_status':'order_status','returns':'returns','billing':'billing','product_info':'product_info'})
compiled_agent = graph.compile()

# Sanity test
test = compiled_agent.invoke({'customer_query': 'Where is my order ORD-1001?'})
print(f"Sanity → category={test['category']} | response[:100]={test['response'][:100]}")


## 6. The Labelled Test Dataset

Each test case carries: `query`, `expected_category` (for L1), `must_contain` keywords (for L3 trajectory), and an `ideal_summary` (for L2 reference). 20 cases across 4 categories. Building a labelled set is the largest one-time cost of evaluation; reusing it for every release is the highest-leverage engineering investment in an LLM project.


In [ ]:
TEST_DATASET = [
    # --- Order Status (5) ---
    {'id':'T01','query':'Where is my order ORD-1001?','expected_category':'order_status',
     'must_contain':['ORD-1001','In Transit','Mumbai'],'ideal_summary':'Tell user the order is in transit to Mumbai with ETA 2 days and tracking link.'},
    {'id':'T02','query':'Has order ORD-1002 been delivered yet?','expected_category':'order_status',
     'must_contain':['ORD-1002','Delivered'],'ideal_summary':'Confirm delivered.'},
    {'id':'T03','query':'Track ORD-1004 please','expected_category':'order_status',
     'must_contain':['ORD-1004','Pending Payment'],'ideal_summary':'Order is on Pending Payment — surface this rather than a tracking number.'},
    {'id':'T04','query':'Status of order ORD-1003?','expected_category':'order_status',
     'must_contain':['ORD-1003','Returned'],'ideal_summary':'Order was returned.'},
    {'id':'T05','query':'Has my Galaxy phone shipped (ORD-1001)?','expected_category':'order_status',
     'must_contain':['ORD-1001','In Transit'],'ideal_summary':'Same as T01 with phone phrasing.'},
    # --- Returns (5) ---
    {'id':'T06','query':'How do I return ORD-1002?','expected_category':'returns',
     'must_contain':['ORD-1002','7-day','5 business days'],'ideal_summary':'Explain pickup + refund timeline.'},
    {'id':'T07','query':'I want a refund for ORD-1003','expected_category':'returns',
     'must_contain':['ORD-1003','Refund','5 business days'],'ideal_summary':'Refund is processed.'},
    {'id':'T08','query':'Return policy on ORD-1001','expected_category':'returns',
     'must_contain':['ORD-1001','7-day','pickup'],'ideal_summary':'Explain 7-day window from delivery.'},
    {'id':'T09','query':'Cancel ORD-1004 I changed my mind','expected_category':'returns',
     'must_contain':['ORD-1004','7-day'],'ideal_summary':'Pre-payment cancel maps to returns flow.'},
    {'id':'T10','query':'When will the refund for ORD-1003 hit my card?','expected_category':'returns',
     'must_contain':['ORD-1003','5 business days'],'ideal_summary':'5 business days.'},
    # --- Billing (5) ---
    {'id':'T11','query':'Why did my UPI fail for ORD-1004?','expected_category':'billing',
     'must_contain':['ORD-1004','Pending Payment','UPI'],'ideal_summary':'Pending payment, complete mandate.'},
    {'id':'T12','query':'I was double-charged for ORD-1001','expected_category':'billing',
     'must_contain':['ORD-1001'],'ideal_summary':'Acknowledge billing concern; ask for screenshot.'},
    {'id':'T13','query':'Show me the invoice amount on ORD-1002','expected_category':'billing',
     'must_contain':['ORD-1002','₹26,990'],'ideal_summary':'Surface invoice amount.'},
    {'id':'T14','query':'Payment status for ORD-1004','expected_category':'billing',
     'must_contain':['ORD-1004','Pending'],'ideal_summary':'Explain Pending Payment.'},
    {'id':'T15','query':'My refund for ORD-1003 is delayed','expected_category':'billing',
     'must_contain':['ORD-1003'],'ideal_summary':'Refund delay falls under billing.'},
    # --- Product Info (5) ---
    {'id':'T16','query':'Tell me about the Samsung Galaxy S24','expected_category':'product_info',
     'must_contain':['Samsung Galaxy S24','₹74,999'],'ideal_summary':'Specs + price.'},
    {'id':'T17','query':'Specs of Sony WH-1000XM5?','expected_category':'product_info',
     'must_contain':['Sony WH-1000XM5','noise cancelling'],'ideal_summary':'Specs.'},
    {'id':'T18','query':'Apple Watch Series 9 price?','expected_category':'product_info',
     'must_contain':['Apple Watch Series 9','₹46,900'],'ideal_summary':'Price + specs.'},
    {'id':'T19','query':'Is the Galaxy S24 in stock?','expected_category':'product_info',
     'must_contain':['Samsung Galaxy S24','In stock'],'ideal_summary':'Confirm stock.'},
    {'id':'T20','query':'Does the WH-1000XM5 have noise cancelling?','expected_category':'product_info',
     'must_contain':['Sony WH-1000XM5','noise cancelling'],'ideal_summary':'Yes.'},
]
print(f'Test dataset: {len(TEST_DATASET)} cases across 4 categories')


## 7. Grader 1 — Routing Accuracy (Deterministic)

Did the router pick the right category? Hard exact-match — no LLM judgment needed. This is the cheapest grader to run and the first you should add to a new agent. If routing is wrong, nothing downstream matters.


In [ ]:
def evaluate_routing(expected_category: str, actual_category: str) -> bool:
    return expected_category == actual_category

print('Running Grader 1: Routing Accuracy')
print('=' * 60)
routing_results = []
for test in TEST_DATASET:
    out = compiled_agent.invoke({'customer_query': test['query']})
    passed = evaluate_routing(test['expected_category'], out['category'])
    routing_results.append({'id': test['id'], 'expected': test['expected_category'],
                            'actual': out['category'], 'passed': passed,
                            'response': out['response']})
    flag = '✅' if passed else '❌'
    print(f"{flag} {test['id']}  expected={test['expected_category']:14s}  actual={out['category']}")

routing_acc = sum(r['passed'] for r in routing_results) / len(routing_results) * 100
print(f'\n→ Routing accuracy: {routing_acc:.0f}% ({sum(r["passed"] for r in routing_results)}/{len(routing_results)})')


## 8. Grader 2 — Response Quality (LLM-as-Judge)

An LLM scores each response on **4 dimensions** (1–5 each): accuracy, completeness, helpfulness, tone. We deliberately use a different model family for the judge (OpenAI judging Anthropic) — same-family judges show ~3-7% inflation when scoring their own family's outputs (self-preference bias). Always log the judge prompt with the score so it's auditable later.


In [ ]:
class QualityScore(BaseModel):
    accuracy: int = Field(ge=1, le=5, description='1-5: correctness of facts')
    completeness: int = Field(ge=1, le=5, description='1-5: did it answer everything')
    helpfulness: int = Field(ge=1, le=5, description='1-5: actionable for the user')
    tone: int = Field(ge=1, le=5, description='1-5: professional + warm')
    rationale: str

judge_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are an expert evaluator for ShopSmart customer-support responses. '
     'Score the response on each rubric dimension from 1 (poor) to 5 (excellent). '
     'Be strict but fair; cite the response text in your rationale.'),
    ('user',
     'CUSTOMER QUERY: {query}\n'
     'IDEAL SUMMARY: {ideal_summary}\n'
     'AGENT RESPONSE: {response}\n\n'
     'Score and explain.')
])
judge_chain = judge_prompt | judge_llm.with_structured_output(QualityScore)

# Sanity: score a deliberately bad response to see the rubric in action
bad = judge_chain.invoke({'query':'Where is my order ORD-1001?',
                          'ideal_summary':'Tell user it is in transit to Mumbai with ETA 2 days.',
                          'response':'I do not know.'})
print(f'Bad-response sanity: accuracy={bad.accuracy} completeness={bad.completeness} helpfulness={bad.helpfulness} tone={bad.tone}')


In [ ]:
print('Running Grader 2: Response Quality (LLM-as-Judge)')
print('=' * 60)
quality_results = []
for i, r in enumerate(routing_results):
    test = TEST_DATASET[i]
    score = judge_chain.invoke({
        'query': test['query'],
        'ideal_summary': test['ideal_summary'],
        'response': r['response'],
    })
    total = score.accuracy + score.completeness + score.helpfulness + score.tone
    quality_results.append({'id':test['id'],'total':total,'breakdown':score.model_dump()})
    print(f"  {test['id']}  total={total:2d}/20  acc={score.accuracy} comp={score.completeness} help={score.helpfulness} tone={score.tone}")

avg_quality = statistics.mean(q['total'] for q in quality_results)
print(f'\n→ Mean response quality: {avg_quality:.1f} / 20  ({avg_quality/20*100:.0f}%)')


## 9. Grader 3 — Trajectory / Tool Correctness (Contains-Check)

Did the agent take the *right steps* — pull the *right* order, cite the *right* policy detail? We approximate this with a contains-check on the labelled `must_contain` keywords. In a real production setup with logged tool-call traces (LangSmith / Langfuse), you'd grade the actual tool-call list against the expected one — but the contains-check on the response is a strong proxy and runs in milliseconds.


In [ ]:
def evaluate_trajectory(response: str, must_contain: list[str]) -> dict:
    matched = [kw for kw in must_contain if kw.lower() in response.lower()]
    precision = len(matched) / max(len(must_contain), 1)
    return {'matched': matched, 'missing': [kw for kw in must_contain if kw not in matched],
            'precision': precision, 'passed': precision >= 0.66}

print('Running Grader 3: Trajectory / Contains-Check')
print('=' * 60)
trajectory_results = []
for i, r in enumerate(routing_results):
    test = TEST_DATASET[i]
    t = evaluate_trajectory(r['response'], test['must_contain'])
    trajectory_results.append({'id':test['id'], **t})
    flag = '✅' if t['passed'] else '⚠️ '
    print(f"{flag} {test['id']}  {t['precision']*100:>3.0f}%  matched={t['matched']}  missing={t['missing']}")

traj_pass = sum(t['passed'] for t in trajectory_results) / len(trajectory_results) * 100
print(f'\n→ Trajectory pass-rate: {traj_pass:.0f}%')


## 10. Combined Eval Suite — Per-Test Table & Per-Category Dashboard

Bring all three graders together. The per-test table is what you triage; the per-category dashboard is what you report.


In [ ]:
print('FULL EVAL SUITE RESULTS')
print('=' * 92)
print(f"{'ID':<5} {'Category':<14} {'Route':<6} {'Quality':<9} {'Traj %':<8} {'Traj Pass':<10}")
print('-' * 92)
for i, test in enumerate(TEST_DATASET):
    r, q, t = routing_results[i], quality_results[i], trajectory_results[i]
    print(f"{test['id']:<5} {test['expected_category']:<14} {'✅' if r['passed'] else '❌':<6} "
          f"{q['total']:>2d}/20    {t['precision']*100:>4.0f}%    {'✅' if t['passed'] else '❌':<10}")


In [ ]:
categories = ['order_status','returns','billing','product_info']
cat_routing = {c: sum(r['passed'] for i,r in enumerate(routing_results)
                      if TEST_DATASET[i]['expected_category']==c) /
                  max(sum(1 for t in TEST_DATASET if t['expected_category']==c),1) * 100 for c in categories}
cat_quality = {c: statistics.mean([q['total'] for i,q in enumerate(quality_results)
                                   if TEST_DATASET[i]['expected_category']==c]) for c in categories}
cat_traj    = {c: sum(t['passed'] for i,t in enumerate(trajectory_results)
                      if TEST_DATASET[i]['expected_category']==c) /
                  max(sum(1 for tt in TEST_DATASET if tt['expected_category']==c),1) * 100 for c in categories}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].bar(categories, [cat_routing[c] for c in categories], color='#2196F3')
axes[0].set_title('L1 — Routing accuracy by category (%)'); axes[0].set_ylim(0,105)
axes[1].bar(categories, [cat_quality[c] for c in categories], color='#9C27B0')
axes[1].set_title('L2 — Mean response quality (/20)'); axes[1].set_ylim(0,21)
axes[2].bar(categories, [cat_traj[c] for c in categories], color='#4CAF50')
axes[2].set_title('L3 — Trajectory pass-rate (%)'); axes[2].set_ylim(0,105)
for ax in axes: ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

weakest_cat = min(categories, key=lambda c: cat_quality[c])
print(f'Weakest category by quality: {weakest_cat}  (mean {cat_quality[weakest_cat]:.1f}/20)')


## 11. pass@k — Measuring Consistency Under Stochasticity

LLMs are non-deterministic even at `temperature=0` (different request batches, different decoding paths). `pass@k` runs each test `k` times and asks: did at least one of the `k` attempts pass? `pass@1` ≈ release accuracy on a single try; `pass@3` ≈ what users see when retries are allowed. The gap between them quantifies brittleness.


In [ ]:
K = 3
subset = TEST_DATASET[:5]
print(f'pass@{K} consistency on {len(subset)} test cases')
print('=' * 60)
pass_at_1, pass_at_k = 0, 0
for test in subset:
    trials = []
    for trial in range(K):
        out = compiled_agent.invoke({'customer_query': test['query']})
        ok = (out['category']==test['expected_category'] and
              evaluate_trajectory(out['response'], test['must_contain'])['passed'])
        trials.append(ok)
    if trials[0]: pass_at_1 += 1
    if any(trials): pass_at_k += 1
    print(f"  {test['id']}  trials={trials}")
print(f'\npass@1 = {pass_at_1}/{len(subset)}    pass@{K} = {pass_at_k}/{len(subset)}')
print('Gap = how often the agent gets it eventually but not first try.')


## 12. Reusable Skills as System Prompts — Why

TechNova (a parallel ShopSmart line-of-business) handles three workloads from one agent: insurance claims, customer support, compliance review. The naive approach is one giant system prompt that lists rules for all three. It's expensive on tokens, hard to update, and the model frequently mixes domains (applying insurance rules to a support query).

**Skills-as-prompts** = each domain is a self-contained instruction pack (system prompt + rules + few-shot). At runtime, classify the query first, then load *only* the relevant skill into the system message. Same base agent, three personalities — and updating one skill never breaks the others.


In [ ]:
SKILLS_REGISTRY = {
    'claim_processing': {
        'name': 'Insurance Claims Processing',
        'system_prompt': (
            'You are a TechNova insurance claims specialist. Process claims with policy validation '
            'and structured intake. Always quote the rupee amount, policy ID, incident date.'),
        'rules': (
            'Rules:\n'
            '1. Confirm the policy ID format INS-YYYY-NNN before processing.\n'
            '2. Surface the claim amount and incident date in the first sentence.\n'
            '3. Cite required documents (FIR/medical/invoice) for the claim type.\n'
            '4. State the expected processing timeline (3 / 5 / 7 working days).')
    },
    'customer_support': {
        'name': 'Customer Support',
        'system_prompt': (
            'You are a TechNova customer-support agent. Be warm, empathetic, and resolve fast. '
            'Acknowledge the customer feeling before stating the action.'),
        'rules': (
            'Rules:\n'
            '1. Empathise in one short sentence first.\n'
            '2. State the resolution path in 2-3 numbered steps.\n'
            '3. End with a concrete next-step (ticket ID, callback time, or self-serve link).')
    },
    'compliance_review': {
        'name': 'Compliance Review',
        'system_prompt': (
            'You are a TechNova compliance reviewer. Identify regulatory issues precisely; '
            'cite the rule short-name; never speculate on penalties.'),
        'rules': (
            'Rules:\n'
            '1. Identify the regulatory regime (RBI / IRDAI / SEBI / DPDPA).\n'
            '2. Cite the specific section or guideline short-name.\n'
            '3. Recommend the next compliance action (filing / disclosure / escalation).\n'
            '4. NEVER state monetary penalty amounts — escalate to legal.')
    },
}
print(f'Skills registry: {list(SKILLS_REGISTRY.keys())}')


## 13. Skill Router + Skill Agent (Dynamic System Prompt Approach)

We use the **dynamic-system-prompt** pattern (more efficient than the tool-call pattern — saves one LLM round-trip). The classifier is keyword-based here for speed; in production you'd use a tiny LLM classifier or a fine-tuned BERT.


In [ ]:
def classify_skill(query: str) -> str:
    q = query.lower()
    if any(k in q for k in ['claim','damage','accident','medical bill','policy holder','insurance']):
        return 'claim_processing'
    if any(k in q for k in ['rbi','irdai','sebi','dpdpa','compliance','regulation','disclosure','section']):
        return 'compliance_review'
    return 'customer_support'

def run_skill_agent(query: str) -> tuple[str, str]:
    skill_key = classify_skill(query)
    skill = SKILLS_REGISTRY[skill_key]
    system_msg = skill['system_prompt'] + '\n\n' + skill['rules']
    msgs = [('system', system_msg), ('user', '{q}')]
    chain = ChatPromptTemplate.from_messages(msgs) | agent_llm | StrOutputParser()
    return chain.invoke({'q': query}), skill_key

# Smoke test
demo_q = 'Process claim INS-2024-456 — water damage to laptop, ₹38,000, on 15 March 2026.'
resp, skill_used = run_skill_agent(demo_q)
print(f'Skill used: {skill_used}')
print(f'Response: {resp[:300]}…')


## 14. A/B Test — Skill-Routed Agent vs Generic Agent

To prove the skills pattern actually helps, we run the same 9 queries through two agents:
1. **Generic agent** — one prompt that lists all three skill domains together.
2. **Skill agent** — classify, then load only the relevant skill.

An LLM-as-Judge scores each response on accuracy / specificity / actionability (1–5 each → 15-point total). Winner = higher mean total.


In [ ]:
AB_QUERIES = [
    {'query':'Process claim INS-2024-456 — water damage to laptop, ₹38,000, on 15 March 2026.','domain':'claim_processing'},
    {'query':'Claim INS-2024-789, accident on 02 April 2026, ₹120,000 vehicle damage.','domain':'claim_processing'},
    {'query':'Process claim INS-2024-321: medical bills ₹54,000 from 10 March hospital admission.','domain':'claim_processing'},
    {'query':'My account is locked, I can\'t log in — please help.','domain':'customer_support'},
    {'query':'I never received my policy document by email; what next?','domain':'customer_support'},
    {'query':'I want to update my registered mobile number; what are the steps?','domain':'customer_support'},
    {'query':'Does our agent-onboarding flow comply with the new IRDAI section on KYC?','domain':'compliance_review'},
    {'query':'A customer\'s data was shared with a vendor — what\'s the DPDPA exposure?','domain':'compliance_review'},
    {'query':'Are we required to disclose the LLM use in our claim-decisioning pipeline under RBI guidelines?','domain':'compliance_review'},
]

GENERIC_PROMPT = ChatPromptTemplate.from_messages([
    ('system','You are TechNova\'s enterprise AI. You handle insurance claims, customer support, and compliance reviews. Be helpful and accurate.'),
    ('user','{q}'),
])
generic_chain = GENERIC_PROMPT | agent_llm | StrOutputParser()

class ABScore(BaseModel):
    accuracy: int = Field(ge=1, le=5)
    specificity: int = Field(ge=1, le=5)
    actionability: int = Field(ge=1, le=5)
    rationale: str

ab_judge_prompt = ChatPromptTemplate.from_messages([
    ('system','Score the response 1-5 on each: accuracy (facts), specificity (concrete vs vague), actionability (clear next step).'),
    ('user','QUERY: {q}\nRESPONSE: {r}\nScore.'),
])
ab_judge = ab_judge_prompt | judge_llm.with_structured_output(ABScore)


In [ ]:
print('Running A/B test  (Generic vs Skill agent, 9 queries)')
print('=' * 78)
ab_results = []
for i, t in enumerate(AB_QUERIES, 1):
    g_resp = generic_chain.invoke({'q': t['query']})
    s_resp, skill_used = run_skill_agent(t['query'])
    g_score = ab_judge.invoke({'q': t['query'], 'r': g_resp})
    s_score = ab_judge.invoke({'q': t['query'], 'r': s_resp})
    g_total = g_score.accuracy + g_score.specificity + g_score.actionability
    s_total = s_score.accuracy + s_score.specificity + s_score.actionability
    ab_results.append({'query':t['query'],'domain':t['domain'],'skill_used':skill_used,
                       'generic_total':g_total,'skill_total':s_total,'winner':'skill' if s_total>g_total else 'generic' if g_total>s_total else 'tie'})
    print(f"  Q{i}  {t['domain']:18s}  generic={g_total:>2d}/15  skill={s_total:>2d}/15  → {ab_results[-1]['winner']}")

g_mean = statistics.mean(r['generic_total'] for r in ab_results)
s_mean = statistics.mean(r['skill_total'] for r in ab_results)
wins = Counter(r['winner'] for r in ab_results)
print(f'\nMean: generic={g_mean:.1f}/15  skill={s_mean:.1f}/15  (Δ={s_mean-g_mean:+.1f})')
print(f'Wins: {dict(wins)}')


## 15. Prompt-Optimization Loop — Fix the Weakest Skill

The A/B test revealed the per-domain breakdown. Identify the weakest skill, ask a meta-LLM to analyse failure patterns and propose rule updates, patch the registry, re-run the same A/B subset, and verify the lift. **This is the eval-driven development loop**: every prompt change is justified by a measurable score delta on a fixed test set, never by vibes.


In [ ]:
domain_scores = {}
for r in ab_results:
    domain_scores.setdefault(r['domain'], []).append(r['skill_total'])
domain_means = {d: statistics.mean(s) for d, s in domain_scores.items()}
for d, m in domain_means.items():
    print(f'  {d:18s} mean skill score = {m:.1f}/15')
worst_domain = min(domain_means, key=domain_means.get)
print(f'\nWeakest domain: {worst_domain}  (mean {domain_means[worst_domain]:.1f}/15)')


In [ ]:
# Have a meta-LLM analyse the worst-domain responses and propose rule patches
worst_examples = [r for r in ab_results if r['domain']==worst_domain]
examples_text = '\n\n'.join(f"QUERY: {e['query']}\nSCORE: {e['skill_total']}/15" for e in worst_examples)
current_rules = SKILLS_REGISTRY[worst_domain]['rules']

opt_prompt = ChatPromptTemplate.from_messages([
    ('system','You are a prompt-engineering expert. Suggest concise additional rules (max 3 bullets) to improve the skill prompt below. Use exactly the same numbered-rule format.'),
    ('user',
     'SKILL: {skill_name}\n\nCURRENT RULES:\n{rules}\n\n'
     'EXAMPLES (low-scoring responses):\n{examples}\n\n'
     'Propose 2-3 additional rules that would have produced higher scores.'),
])
opt_chain = opt_prompt | agent_llm | StrOutputParser()
patch = opt_chain.invoke({'skill_name': worst_domain, 'rules': current_rules, 'examples': examples_text})
print('---- Proposed rule patch ----')
print(patch)


In [ ]:
# Apply the patch to a V2 registry, re-run that domain's queries, compare
SKILLS_REGISTRY_V2 = copy.deepcopy(SKILLS_REGISTRY)
SKILLS_REGISTRY_V2[worst_domain]['rules'] += '\n\n# Optimisation patch (auto-generated):\n' + patch

def run_skill_agent_v(registry, query):
    skill_key = classify_skill(query)
    skill = registry[skill_key]
    system_msg = skill['system_prompt'] + '\n\n' + skill['rules']
    chain = ChatPromptTemplate.from_messages([('system', system_msg),('user','{q}')]) | agent_llm | StrOutputParser()
    return chain.invoke({'q': query}), skill_key

v1_scores, v2_scores = [], []
print(f'Re-running {worst_domain} queries on V1 vs V2 of the skill')
print('=' * 70)
for t in [q for q in AB_QUERIES if q['domain']==worst_domain]:
    r1,_ = run_skill_agent_v(SKILLS_REGISTRY,    t['query'])
    r2,_ = run_skill_agent_v(SKILLS_REGISTRY_V2, t['query'])
    s1 = ab_judge.invoke({'q':t['query'],'r':r1})
    s2 = ab_judge.invoke({'q':t['query'],'r':r2})
    t1 = s1.accuracy+s1.specificity+s1.actionability
    t2 = s2.accuracy+s2.specificity+s2.actionability
    v1_scores.append(t1); v2_scores.append(t2)
    print(f'  V1={t1:>2d}/15  V2={t2:>2d}/15  Δ={t2-t1:+d}')

if v1_scores:
    print(f'\nMean: V1={statistics.mean(v1_scores):.1f}/15  V2={statistics.mean(v2_scores):.1f}/15  '
          f'(Δ={statistics.mean(v2_scores)-statistics.mean(v1_scores):+.1f})')


## 16. Conclusion & Key Takeaways

We built a **3-layer eval suite** for ShopSmart's support agent and used it to drive a **measurable prompt-optimization loop** for TechNova's skills library:

- **L1 routing** is deterministic and cheap — always run it on every release.
- **L2 LLM-as-Judge quality** uses a different-family judge to reduce self-preference bias.
- **L3 trajectory** uses a `must_contain` proxy until you have full tool-call traces from LangSmith / Langfuse.
- **pass@k** quantifies brittleness — gap between `pass@1` and `pass@k` tells you how often retries save users.
- **Reusable skills** beat one-giant-prompt: cleaner updates, lower per-query token cost, easier debugging.
- **A/B + optimization loop** turns prompt engineering from craft into measurable engineering.

Every metric you see here would normally be logged to LangSmith or Langfuse (Labs 9.1 / 9.2) — and the trace IDs let you correlate a per-query failure back to the exact tool call that misfired.


## 17. Stretch Exercise (Optional)

1. **Add a 4th grader: latency.** Time each `compiled_agent.invoke(...)` call and produce a per-category P50 / P95 latency report. Is the weakest-quality category also the slowest?
2. **Cross-judge calibration.** Re-run Grader 2 with Claude as the judge (instead of GPT). Compute the per-test score delta — how much does the judge family swing the result? Document the bias.
3. **Skills-as-tools variant.** Re-implement the skill loader as a `load_skill` tool the agent calls explicitly (vs the dynamic-system-prompt pattern). Compare token cost and latency.
4. **Auto-generate test cases.** Use Claude to generate 10 new `must_contain`-style test cases from a template — manually review them — add them to the dataset and verify the suite still passes.


## 18. Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. What are the 3 layers of agent eval and what does each catch?**

*Hint:* Each layer answers a question no other layer can answer.

*Answer sketch:* L1 routing — did we pick the right tool/category (catches mis-route). L2 response quality (LLM-as-Judge) — is the answer accurate, complete, helpful, well-toned (catches bad content even when routing is right). L3 trajectory / tool correctness — did we actually use the right inputs (catches confidently wrong answers from looking up the wrong record).

---

**Q2. Routing accuracy vs response quality — why eval them separately?**

*Hint:* They have different failure modes and different fix recipes.

*Answer sketch:* Routing is binary and deterministic — fixes are router-prompt or fine-tuning the classifier. Response quality is multi-dimensional and judge-scored — fixes are handler-prompt edits, retrieval improvements, output schema. Combining them hides which side is broken.

---

**Q3. What is 'trajectory correctness' and why does it differ from response quality?**

*Hint:* A response can read perfectly while the agent looked up the wrong record.

*Answer sketch:* Trajectory = did the agent take the right steps with the right inputs (right tool, right args, right order). Quality grades the surface answer; trajectory grades the path. An agent that confidently quotes the wrong order ID's status passes quality and fails trajectory.

---

**Q4. What is pass@k and when do you use it?**

*Hint:* It quantifies stochastic brittleness.

*Answer sketch:* pass@k = fraction of test cases where at least 1 of k attempts passes. pass@1 ≈ release accuracy on a single try; pass@k ≈ what users see when retries are allowed. Use it whenever the agent is non-deterministic (always, for LLMs at temp>0 and even at 0). The gap quantifies brittleness.

---

**Q5. How do you build a labelled test set for routing accuracy without hand-labelling?**

*Hint:* Mine production traces.

*Answer sketch:* Pull queries from LangSmith / Langfuse traces, cluster by category-the-agent-picked, sample evenly across clusters, and have a small panel (or a stronger LLM) review only the disputed ones. Auto-label the unanimous bulk; hand-label the ambiguous ~10%. This bootstraps a labelled set for ~10x less effort than from scratch.

---

**Q6. LLM-as-Judge biases — how do you calibrate?**

*Hint:* Different family, different position, different anchors.

*Answer sketch:* Use a different model family for the judge than the actor (self-preference bias). Randomise option order if the judge picks 'A vs B' (position bias). Define explicit numeric anchors per rubric dimension (`5=cites both order ID and ETA; 1=missing both`). Periodically calibrate against a small human-rated gold set; track judge-vs-human agreement.

---

**Q7. What is a 'reusable skill' in the prompt-optimization context?**

*Hint:* A skill = a self-contained instruction pack the agent loads on demand.

*Answer sketch:* A skill is a versioned dict `{name, system_prompt, rules, few_shot}` that the runtime loads into the system message *only* when the classifier picks that domain. One base agent → many personalities. Updating one skill never breaks another. Compare to one-giant-prompt: cheaper tokens, easier diffs, A/B-able per skill.

---

**Q8. How do you A/B test a prompt change in a production agent?**

*Hint:* Hold the test set fixed; change one prompt; compare scores.

*Answer sketch:* Freeze a labelled test set. Run V1 (current) and V2 (candidate) on the same set. Score with the same judge. Compare per-test deltas (paired) and the mean. Ship V2 only if mean improves AND no per-test regression > X. In live traffic, dark-launch V2 to ε% of users with the same scoring pipeline before full rollout.
